# DCT Laboratory — Volume I, Chapter 10
## Enterprise Capital Architecture
**Seed `26110`** · Companion to Chapter 10 and **AXIOM Lab 1.10** (module **AXIOM-10**) · Mirrored in `DCT_V1_Ch10_Lab.xlsx`

This laboratory reproduces **Worked Example 10.1, *Keep the Engineers or Rebuild Them?***, with the book's
declarations, and works the five steps of AXIOM Lab 1.10, *Meridian's Capital Ledger*: build the capital
vector, price the rent, test the asymmetry, allocate a budget, audit the ledger. Every Meridian number below
is the book's or follows from its declarations (Worked Example 10.1, Table 10.5, Figure 10.3, Online Appendix
10.B.6); the final cell checks the printed ones. Money is in \$ million.

## The declarations (Worked Example 10.1, Steps 1–2)

The capital vector of Equation 10.1 has **ten forms** (Table 10.1): financial, human, intellectual,
technological, organizational, relational, strategic, governance, innovation and informational capital.
Meridian's restructuring sells part of Industrial Systems on the terms of Worked Example 3.1: \$112 million net
if the shared engineering team goes with the unit, \$92 million if Meridian keeps it. The unit takes 10 percent
of the technology position, and without the team the workforce-capability index falls from 58 to 53, below the
floor of 55 that the digital build and the automation program require. Capability is a capital stock that loses
2.5 percent a quarter to attrition and the aging of skills, $C' = 0.975\,C + I$ (Equation 10.3); hiring and
training add at most 2.5 points a quarter, at about \$4 million a point, the price the buyer's discount implies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import log
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26110

# Table 10.1: the ten forms of the capital vector (Equation 10.1): form, symbol, content, typical carrier
FORMS = [("Financial", "C_F", "money and funding capacity", "treasury"),
         ("Human", "C_H", "skills embodied in the workforce", "people, functions"),
         ("Intellectual", "C_I", "codified knowledge and IP", "IP estate; data"),
         ("Technological", "C_T", "installed systems and platforms", "operations; IT"),
         ("Organizational", "C_O", "routines and coordination capacity", "the wiring itself"),
         ("Relational", "C_R", "external trust and commitments", "boundary components"),
         ("Strategic", "C_S", "positional advantage and options", "corporate center"),
         ("Governance", "C_G", "decision quality and legitimacy", "board; governance"),
         ("Innovation", "C_N", "pipeline of unrealized options", "R&D; ventures"),
         ("Informational", "C_Info", "decision-relevant data and models", "platforms; functions")]

# Worked Example 10.1 (the sale's terms are Worked Example 3.1's); money in $ million
LIQUIDITY, TECH, TECH_SHARE = 620.0, 41.0, 0.10     # liquidity, technology index, share of it the unit takes
PROCEEDS_LEAVE, DISCOUNT = 112.0, 20.0               # net proceeds if the team goes; the discount to keep it
C_KEEP, POINTS_LOST, FLOOR = 58.0, 5.0, 55.0         # capability with the team, points lost without it, floor
DELTA, CAP = 0.025, 2.5                              # attrition a quarter; most hiring and training a quarter
PROCEEDS_KEEP, C_LEAVE = PROCEEDS_LEAVE - DISCOUNT, C_KEEP - POINTS_LOST
PRICE = DISCOUNT / POINTS_LOST                       # $ million a capability point
CBAR, RENT = CAP / DELTA, DELTA * C_KEEP             # ceiling under the cap; rent on 58 (points a quarter)

def path(c0, invest, n, d=DELTA, target=None):
    # C' = (1 - d) C + I for n quarters (Equation 10.3); invest is a number or one per quarter;
    # with a target, invest only what lands the stock on it, never overshooting
    c, spend = [c0], []
    for k in range(n):
        i = float(invest[k]) if np.ndim(invest) else float(invest)
        if target is not None:
            i = min(i, max(target - (1 - d) * c[-1], 0.0))
        spend.append(i); c.append((1 - d) * c[-1] + i)
    return np.array(c), np.array(spend)

def build_time(c0, c1, cap=CAP, d=DELTA):
    # Theorem 10.3(iii): no spending pattern inside the cap gets from c0 to c1 sooner
    cbar = cap / d
    return log((cbar - c0) / (cbar - c1)) / log(1 / (1 - d))

def first_at(c, level):
    # the first quarter at or above level (None if never)
    hit = np.nonzero(np.asarray(c) >= level - 1e-9)[0]
    return int(hit[0]) if len(hit) else None

print(f"capital vector: {len(FORMS)} forms (Table 10.1)")
print(f"capability decays {DELTA:.1%} a quarter, {1 - (1 - DELTA)**4:.1%} a year; hiring and training at most {CAP} points a quarter")
print(f"price of a point: ${DISCOUNT:.0f} million for {POINTS_LOST:.0f} points = ${PRICE:.1f} million")

## Panel 1 — Build the capital vector (Table 10.1; Worked Example 10.1, Step 1)

AXIOM Lab 1.10, step 1. The capital vector carries ten forms. Definitions 10.3–10.9 define the first of them one
by one, and Table 10.2 profiles those on the four properties that drive their dynamics; the other three,
governance, innovation and informational capital, are treated with the same machinery. Meridian's declared state carries four capital
readings (Table 5.2, tagged ECA), each with a carrier, a unit and an error bar. It declares no proxy for
organizational capital: Section 4.5 counts organizational cohesion among the latent readings, and Section 10.4
warns that the softer forms have the widest error bars exactly where they matter most. Then read the sale as a
**conversion** (Theorem 10.1): technological, human and relational capital into cash, with the buyer's
discount as a market quote for a stock that never appears on the balance sheet, **\$4 million a point**.

In [ ]:
print(f"Table 10.1: the {len(FORMS)} forms of the capital vector")
print(f"  {'form':15s} {'symbol':7s} {'content':36s} typical carrier")
for name, sym, content, carrier in FORMS:
    print(f"  {name:15s} {sym:7s} {content:36s} {carrier}")

# Table 10.2: the forms of Definitions 10.3-10.9 on the four properties (delta a year)
PROFILE = {"C_F": ("about 0", "immediate", "fully", "fully"),
           "C_H": ("0.10-0.25", "years", "at the margin", "no"),
           "C_I": ("0.05-0.20", "years", "partly", "yes"),
           "C_T": ("0.15-0.30", "quarters to years", "in parts", "partly"),
           "C_O": ("0.03-0.10", "years", "no", "no"),
           "C_R": ("0.05-0.15, plus breach", "years", "no", "no"),
           "C_S": ("tied to the regime", "years", "rarely", "partly")}
print("\nTable 10.2: the four properties, for the forms of Definitions 10.3-10.9 (the other three are not profiled)")
print(f"  {'form':15s} {'delta a year':23s} {'build time':18s} {'tradable':14s} separable")
for name, sym, _, _ in FORMS:
    if sym in PROFILE:
        d, b, t, s = PROFILE[sym]
        print(f"  {name:15s} {d:23s} {b:18s} {t:14s} {s}")

# Meridian's capital readings: Table 5.2 (tagged ECA), carriers from Tables 9.1 and 10.1
READINGS = [("x1 liquidity", "financial", "$620 million", "treasury systems, +/-1%", "treasury (TRS)"),
            ("x2 net leverage", "financial", "3.4x (covenant 4.5x)", "audited accounts, +/-2%", "treasury (TRS)"),
            ("x3 workforce capability", "human", "58 (index)", "skills survey, +/-5", "engineering (ENG)"),
            ("x4 technology platform", "technological", "41 (index)", "architecture audit, +/-4", "operations; IT"),
            ("none declared", "organizational", "latent", "no declared proxy", "the wiring itself")]
print("\nMeridian's capital vector as declared (Table 5.2)")
print(f"  {'reading':24s} {'form':15s} {'today':21s} {'measured by (error)':25s} carrier")
for r in READINGS:
    print(f"  {r[0]:24s} {r[1]:15s} {r[2]:21s} {r[3]:25s} {r[4]}")

TECH_AFTER = TECH * (1 - TECH_SHARE)
AFTER = {"keep the team": (LIQUIDITY + PROCEEDS_KEEP, C_KEEP, TECH_AFTER),
         "let it go": (LIQUIDITY + PROCEEDS_LEAVE, C_LEAVE, TECH_AFTER)}
print("\nThe sale as a conversion (Theorem 10.1): technological, human and relational capital into cash")
print(f"  {'':24s} {'today':>7s} {'keep the team':>15s} {'let it go':>11s}")
for j, label in enumerate(("liquidity ($ million)", "workforce capability", "technology platform")):
    today = (LIQUIDITY, C_KEEP, TECH)[j]
    print(f"  {label:24s} {today:7.1f} {AFTER['keep the team'][j]:15.1f} {AFTER['let it go'][j]:11.1f}")
print(f"the buyer's discount: ${PROCEEDS_LEAVE - PROCEEDS_KEEP:.0f} million for {C_KEEP - C_LEAVE:.0f} capability points"
      f" = ${(PROCEEDS_LEAVE - PROCEEDS_KEEP) / (C_KEEP - C_LEAVE):.1f} million a point")

## Panel 2 — Price the rent (Theorem 10.3(ii); Figure 10.1; Worked Example 10.1, Step 2)

AXIOM Lab 1.10, step 2, first part: hold each stock at its target with its sustaining flow. Under constant
investment $I$ a stock converges to $I/\delta$, so holding $C^{\ast}$ needs the perpetual flow $I = \delta C^{\ast}$,
and any lower flow guarantees decline: capital is a rental. Figure 10.1's three stocks share the target 100 and
decay at 5, 10 and 25 percent a year. From zero they take **44.9, 21.9 and 8.0 years** to reach 90 percent of the
target at the rent, and **11.7, 5.7 and 2.1** at twice the rent. Meridian's capability rent at 58 is
**1.45 points a quarter, about \$23 million a year**, already in the operating budget.

In [ ]:
RENTAL = [(0.05, "organizational", "#1F4E79"), (0.10, "human", "#E65100"), (0.25, "technological", "#B71C1C")]

def years_to(share, invest, d, target=100.0):
    # from zero at constant investment, C_k = (I/d)(1 - (1 - d)^k): years to reach share x target
    cinf = invest / d
    return log(cinf / (cinf - share * target)) / log(1 / (1 - d))

print("Figure 10.1: three stocks with the target 100, built from zero, yearly")
print(f"  {'stock':15s} {'delta':>6s} {'rent a year':>12s} {'years to 90%: at the rent':>26s} {'at twice the rent':>18s} {'(1/delta) ln 10':>16s}")
rental = {}
for d, name, _ in RENTAL:
    rental[name] = (years_to(0.9, d * 100, d), years_to(0.9, 2 * d * 100, d))
    print(f"  {name:15s} {d:6.2f} {d * 100:12.0f} {rental[name][0]:26.1f} {rental[name][1]:18.1f} {log(10) / d:16.1f}")

print(f"\nMeridian's capability: {DELTA:.1%} a quarter is {1 - (1 - DELTA)**4:.1%} a year, inside the 6-11% attrition range of Table 4.3")
print(f"rent on 58: {DELTA} x 58 = {RENT:.2f} points a quarter ({4 * RENT:.1f} a year)"
      f" = ${PRICE * RENT:.1f} million a quarter, ${4 * PRICE * RENT:.1f} million a year")
held, held_spend = path(C_KEEP, RENT, 8)
print(f"held with its sustaining flow for eight quarters: capability {held.min():.2f} to {held.max():.2f}, spending {held_spend.sum():.1f} points")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
T = 30
for d, name, col in RENTAL:
    at_rent, _ = path(0.0, d * 100, T, d)
    fast, _ = path(0.0, 2 * d * 100, T, d, target=100.0)
    ax.plot(np.arange(T + 1), at_rent, color=col, lw=2, label=f"{name}, δ = {d:.0%}: rent {d * 100:.0f} a year")
    ax.plot(np.arange(T + 1), fast, color=col, lw=1.2, ls="--")
ax.axhline(90, color="#999999", lw=0.8, ls=":")
ax.text(T, 88, "90% of target", ha="right", va="top", fontsize=8, color="#616161")
ax.plot([], [], color="#616161", lw=1.2, ls="--", label="dashed: twice the rent until the target")
ax.set(xlabel="years from zero", ylabel="stock (target = 100)", xlim=(0, T), ylim=(0, 108),
       title="At the rent, slow stocks take decades to build (seed 26110)")
ax.legend(frameon=False, fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

## Panel 3 — Run the rebuild clock (Theorem 10.3(iii); Figure 10.3; Worked Example 10.1, Steps 3–4)

With the cap at 2.5 points a quarter the ceiling is $\bar C = 2.5/0.025 = 100$, and no spending pattern reaches
a target sooner than full throttle: from 53, at least $\ln(47/45)/\ln(1/0.975) = 1.7$ quarters to clear 55 and
$\ln(47/42)/\ln(1/0.975) = 4.4$ to regain 58, **two quarters and five** in practice. At the old maintenance
budget capability creeps back toward 58 and clears 55 only after **21 quarters**, more than five years.
Step 4: while capability is below 55 the digital build and the automation program are off the menu
(Propositions 10.3 and 10.4); the sale removes them on the day it closes.

In [ ]:
full, spend_full = path(C_LEAVE, CAP, 24, target=C_KEEP)      # rebuild at the cap, landing on 58
maint, _ = path(C_LEAVE, RENT, 120)                            # the old maintenance budget
t55, t58 = build_time(C_LEAVE, FLOOR), build_time(C_LEAVE, C_KEEP)
k55, k58 = first_at(full, FLOOR), first_at(full, C_KEEP)
k55m, k57m = first_at(maint, FLOOR), first_at(maint, 57.0)
print(f"ceiling under the cap: {CAP} / {DELTA} = {CBAR:.0f}")
print(f"full throttle from 53: at least {t55:.2f} quarters to clear 55 and {t58:.2f} to regain 58")
fmt = lambda xs, nd=2: ", ".join(f"{v:.{nd}f}" for v in np.round(xs, nd))
print(f"capability at the cap, quarters 0-{k58}: {fmt(full[:k58 + 1])} -> clears 55 at quarter {k55}, regains 58 at quarter {k58}")
print(f"investment at the cap: {fmt(spend_full[:k58])} points a quarter")
print(f"old maintenance budget ({RENT:.2f} a quarter): capability = 58 - 5 x 0.975^k;"
      f" clears 55 at quarter {k55m} ({k55m / 4:.2f} years), {maint[24]:.2f} after six years, 57 only at quarter {k57m}")
print(f"Step 4: the digital build and the automation program are off the menu for {k55} quarters at the cap, {k55m} at the old budget")

q = np.arange(25)
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.axvspan(0, k55, color="#F6E3E3", lw=0)
ax.axhline(FLOOR, color="#B71C1C", lw=0.9, ls="--")
ax.text(6.3, FLOOR + 0.08, "floor 55: digital build and automation", ha="left", va="bottom", fontsize=8, color="#B71C1C")
ax.plot(q, np.full(len(q), C_KEEP), color="#1F4E79", lw=3, alpha=0.5, label=f"keep the team: proceeds \\${PROCEEDS_KEEP:.0f} million")
ax.step(q, full[:25], where="post", color="#E65100", lw=2, label=f"let it go, rebuild at the cap: \\${PROCEEDS_LEAVE:.0f} million; {k55} quarters below 55")
ax.step(q, maint[:25], where="post", color="#616161", lw=1.6, ls=":", label=f"let it go, old maintenance budget: {k55m} quarters below 55")
ax.set(xlabel="quarters after closing", ylabel="workforce capability (index)", xlim=(0, 24), ylim=(52.3, 60.6),
       title="Rebuilding at the cap clears the floor in two quarters (seed 26110)")
ax.legend(frameon=False, fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()

## Panel 4 — Cut the rent for four quarters (Proposition 10.4; Section 11.8)

AXIOM Lab 1.10, step 2, second part: cut investment for four quarters and watch the decline, and the recovery at
the old budget and at the cap; this is the proposal the transformation office must answer. The book prints its
first consequence in Chapter 11 (Section 11.8): skipping the rent adds about \$23 million a year to reported
EBITDA, while capability falls below the floor of 55 **within three quarters**. The rest follows from the
declarations of Worked Example 10.1.

In [ ]:
CUT = 4
cut, _ = path(C_KEEP, 0.0, CUT)
below = int(np.argmax(cut < FLOOR))
print(f"capability with no investment, quarters 0-{CUT}: {fmt(cut)} -> below 55 from quarter {below}")
print(f"the cut saves {CUT * RENT:.2f} points = ${PRICE * CUT * RENT:.1f} million, the year's rent")
c4 = cut[-1]
old, _ = path(c4, RENT, 400)
fast, fast_spend = path(c4, CAP, 40, target=C_KEEP)
r55, r58 = first_at(fast, FLOOR), first_at(fast, C_KEEP)
print(f"recovery from {c4:.2f} at the old budget: clears 55 after {first_at(old, FLOOR)} quarters and creeps toward 58"
      f" without regaining it ({old[40]:.2f} after ten years)")
print(f"recovery at the cap: clears 55 after {r55} quarters, regains 58 after {r58} (Theorem 10.3(iii): at least {build_time(c4, C_KEEP):.2f})")
rebuy, hold = fast_spend[:r58].sum(), RENT * r58
print(f"over those {r58} quarters the cap invests {rebuy:.2f} points (${PRICE * rebuy:.1f} million) against {hold:.2f}"
      f" (${PRICE * hold:.1f} million) at the rent: ${PRICE * (rebuy - hold):.1f} million more")
print(f"capability is below 55 for {CUT - below + 1 + r55 - 1} quarters (quarters {below}-{CUT + r55 - 1}): the digital build waits a year")

## Panel 5 — Test the asymmetry (Proposition 10.4; Online Appendix 10.B.2)

AXIOM Lab 1.10, step 3: destroy 30 percent of human capital in one quarter and count the quarters needed to
recover, here from the declarations of Worked Example 10.1. Burning capital is fast; rebuilding it is limited by
$\delta$ (Proposition 10.4). Online Appendix 10.B.2 prices a write-off
$w$ from a floor $c$: climbing back takes at least
$k(w) = \ln\bigl((\bar C - c + w)/(\bar C - c)\bigr)/\ln\bigl(1/(1-\delta)\bigr)$ periods, about
$w/(\delta(\bar C - c))$ for small write-offs. For Meridian's 2 points below the floor that is **1.72 quarters**
exactly and **1.78** by the approximation.

In [ ]:
SHOCK = 0.30
c0 = (1 - SHOCK) * C_KEEP
s_cap, s_spend = path(c0, CAP, 40, target=C_KEEP)
s_old, _ = path(c0, RENT, 400)
k55s, k58s = first_at(s_cap, FLOOR), first_at(s_cap, C_KEEP)
print(f"destroy {SHOCK:.0%} of capability in one quarter: 58 -> {c0:.1f}, a write-off of {C_KEEP - c0:.1f} points")
print(f"at the cap: clears 55 after {k55s} quarters, regains 58 after {k58s} (Theorem 10.3(iii): at least {build_time(c0, C_KEEP):.2f}),"
      f" investing {s_spend[:k58s].sum():.1f} points (${PRICE * s_spend[:k58s].sum():.0f} million)")
print(f"at the old budget: clears 55 only after {first_at(s_old, FLOOR)} quarters and never regains 58")

def repurchase(w, c, d=DELTA, cbar=CBAR):
    # Online Appendix 10.B.2: the fewest periods back to c after a write-off w, and its approximation
    return log((cbar - c + w) / (cbar - c)) / log(1 / (1 - d)), w / (d * (cbar - c))

for w, c, label in ((2.0, FLOOR, "2 points below the floor of 55"), (C_KEEP - c0, C_KEEP, "the 30% write-off from 58")):
    ex, ap = repurchase(w, c)
    print(f"repurchase period, {label}: {ex:.2f} quarters exactly, {ap:.2f} by w/(delta(Cbar - c))")

q = np.arange(41)
cut_cap = np.concatenate([cut, fast[1:]])[:41]
cut_old = np.concatenate([cut, old[1:]])[:41]
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.axhline(FLOOR, color="#B71C1C", lw=0.9, ls="--")
ax.axhline(C_KEEP, color="#999999", lw=0.8)
ax.step(q, s_cap[:41], where="post", color="#E65100", lw=2, label=f"destroy 30% in a quarter, rebuild at the cap: back in {k58s}")
ax.step(q, s_old[:41], where="post", color="#E65100", lw=1.4, ls=":", label="destroy 30%, old budget: never back")
ax.step(q, cut_cap, where="post", color="#1F4E79", lw=2, label=f"cut for four quarters, then the cap: back after {r58} more")
ax.step(q, cut_old, where="post", color="#1F4E79", lw=1.4, ls=":", label="cut for four quarters, then the old budget")
ax.text(15.5, FLOOR + 0.2, "floor 55", ha="left", va="bottom", fontsize=8, color="#B71C1C")
ax.set_yticks([40, 45, 50, 55, 58])
ax.set(xlabel="quarters", ylabel="workforce capability (index)", xlim=(0, 40), ylim=(39, 60),
       title="One quarter to destroy, fourteen to rebuild (seed 26110)")
ax.legend(frameon=False, fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

## Panel 6 — Compare the cash (Table 10.5; Online Appendix 10.B.6)

Worked Example 10.1, Step 5. Over the five-quarter rebuild window, full-throttle rebuilding invests **11.9 points
(\$47.6 million)** against the **7.25 points (\$29.0 million)** needed to hold 58 with the team in place:
\$18.6 million more. Net of the extra \$20 million of proceeds, rebuilding comes out \$1.4 million ahead
(Table 10.5): in cash, the two options are a wash. Online Appendix 10.B.6 shows the comparison is robust to the
cap and the attrition rate, and prices a \$30 million ask.

In [ ]:
def compare(cap=CAP, d=DELTA, ask=DISCOUNT):
    # rebuild at the cap from 53 against holding 58 with the team, over the rebuild window
    reb, spend = path(C_LEAVE, cap, 200, d, target=C_KEEP)
    q55, q58 = first_at(reb, FLOOR), first_at(reb, C_KEEP)
    inv_rebuild, inv_keep = spend[:q58].sum(), d * C_KEEP * q58
    net_keep, net_rebuild = PROCEEDS_LEAVE - ask - PRICE * inv_keep, PROCEEDS_LEAVE - PRICE * inv_rebuild
    return dict(q55=q55, q58=q58, rebuild=inv_rebuild, keep=inv_keep, extra=PRICE * (inv_rebuild - inv_keep),
                net_keep=net_keep, net_rebuild=net_rebuild)

base = compare()
TABLE105 = [("Proceeds at closing ($ million)", PROCEEDS_KEEP, PROCEEDS_LEAVE, "0f"),
            ("Capability at closing", C_KEEP, C_LEAVE, "0f"),
            ("Quarters below the floor of 55", 0, base["q55"], "0f"),
            ("Quarters below 58", 0, base["q58"], "0f"),
            ("Capability investment, five quarters (points)", base["keep"], base["rebuild"], "2f"),
            ("Capability investment, five quarters ($ million)", PRICE * base["keep"], PRICE * base["rebuild"], "1f"),
            ("Proceeds net of that investment ($ million)", base["net_keep"], base["net_rebuild"], "1f")]
print(f"Table 10.5   {'':38s} {'keep the team':>14s} {'let it go, rebuild':>19s}")
for label, keep, leave, f in TABLE105:
    print(f"  {label:49s} {keep:14{'.' + f}} {leave:19{'.' + f}}")
print(f"rebuilding invests {base['rebuild'] - base['keep']:.2f} points (${base['extra']:.1f} million) more,"
      f" and comes out ${base['net_rebuild'] - base['net_keep']:.1f} million ahead net of the extra proceeds")

CASES = [("as declared", CAP, DELTA), ("cap 2.0 points a quarter", 2.0, DELTA),
         ("attrition 6% a year", CAP, 1 - 0.94 ** 0.25), ("attrition 11% a year", CAP, 1 - 0.89 ** 0.25)]
print(f"\nOnline Appendix 10.B.6   {'delta a quarter':>16s} {'clears 55':>10s} {'regains 58':>11s} {'extra cost':>11s}")
sens = {}
for label, cap, d in CASES:
    sens[label] = compare(cap, d)
    r = sens[label]
    print(f"  {label:25s} {d:14.4f} {r['q55']:10d} {r['q58']:11d} {r['extra']:11.1f}")
ask30 = compare(ask=30.0)
print(f"a $30 million ask: keeping nets ${ask30['net_keep']:.1f} million against ${ask30['net_rebuild']:.1f} million for rebuilding,"
      f" ${ask30['net_rebuild'] - ask30['net_keep']:.1f} million more in cash: the price the board would put on two quarters of digital-program time")

## Panel 7 — Allocate a budget (Theorems 10.2(i) and 10.4; Figure 10.2)

AXIOM Lab 1.10, step 4. Section 10.7's technology company needs technology, people and routines in fixed
proportions, $F = \min(C_T/1.0,\ C_H/0.5,\ C_O/0.75)$, with stocks 80, 45 and 60. The ratios are 80, 90 and 80:
technology and routines tie as the bottleneck and human capital has 12.5 percent slack. Funding the bottleneck
raises capacity by **0.57 a point** until all three tie at 90 after 17.5 points, and by **0.44** after; these
slopes are the budget's shadow value $\lambda$ (Theorem 10.4). An equal split earns 0.33 a point, and the
"talent war" buys nothing: capacity after 30 points is **95.6, 90.0 and 80.0**. Capacity here reads no financial
capital, so a finance-heavy split adds nothing until the money is converted into one of the three forms.

In [ ]:
STOCKS = np.array([80.0, 45.0, 60.0])        # technology, human, organizational (Figure 10.2)
REQ = np.array([1.0, 0.5, 0.75])             # requirements r_i: F = min_i C_i / r_i
NAMES = ("technology", "human", "organizational")

def capacity(c, r=REQ):
    return float(np.min(np.asarray(c) / r))

def need(level, c=STOCKS, r=REQ):
    # Theorem 10.4(ii): a_i(l) = max(r_i l - C_i, 0), the funding that lifts every form to level l
    return np.maximum(r * level - c, 0.0)

def fund_bottleneck(B, c=STOCKS, r=REQ):
    # the level l_B with sum_i a_i(l_B) = B, walking the kinks where another form joins the tie
    kinks = np.unique(c / r)
    level = kinks[0]
    for nxt in list(kinks[1:]) + [np.inf]:
        tied = r[c / r <= level + 1e-9].sum()
        spent = need(level, c, r).sum()
        if spent + (nxt - level) * tied >= B - 1e-12:
            level += (B - spent) / tied
            break
        level = nxt
    return level, need(level, c, r)

POLICIES = {"fund the bottleneck": lambda B: fund_bottleneck(B)[0],
            "equal split": lambda B: capacity(STOCKS + B / 3),
            "all to human capital": lambda B: capacity(STOCKS + np.array([0.0, B, 0.0]))}
ratios = STOCKS / REQ
print("ratios C/r:", ", ".join(f"{n} {v:.0f}" for n, v in zip(NAMES, ratios)),
      f"-> capacity {capacity(STOCKS):.0f}; human capital's slack {ratios[1] / ratios.min() - 1:.1%}")
tie = need(ratios.max()).sum()
lam = [1 / REQ[ratios <= ratios.min() + 1e-9].sum(), 1 / REQ.sum()]
print(f"fund the bottleneck: lambda = 1/(1.0 + 0.75) = {lam[0]:.3f} a point until all three tie at {ratios.max():.0f}"
      f" after {tie:.1f} points, then 1/(1.0 + 0.5 + 0.75) = {lam[1]:.3f}")
print(f"{'budget':>8s}" + "".join(f"{p:>24s}" for p in POLICIES))
for B in (0.0, 10.0, 17.5, 30.0):
    print(f"{B:8.1f}" + "".join(f"{f(B):24.2f}" for f in POLICIES.values()))
level30, alloc30 = fund_bottleneck(30.0)
print("the bottleneck allocation of 30 points:", ", ".join(f"{n} {v:.2f}" for n, v in zip(NAMES, alloc30)), f"-> capacity {level30:.2f}")
eq_rate = (POLICIES["equal split"](30.0) - capacity(STOCKS)) / 30.0
print(f"equal split: {eq_rate:.3f} a point; all to human capital: {POLICIES['all to human capital'](30.0) - capacity(STOCKS):.1f} in all")

# Online Appendix 10.B.3: F = min(C1/1, C2/2), stocks (10, 30), budget 20
l3, a3 = fund_bottleneck(20.0, np.array([10.0, 30.0]), np.array([1.0, 2.0]))
print(f"Online Appendix 10.B.3: capacity {l3:.0f}, allocation ({fmt(a3, 1)}), kink after"
      f" {need(15.0, np.array([10.0, 30.0]), np.array([1.0, 2.0])).sum():.0f} units")
# Online Appendix 10.B.4: complements F = C1^a C2^b; the optimal final stocks are in the ratio a : b
for a, b in ((0.3, 0.3), (0.4, 0.2)):
    tgt = np.array([a, b]) / (a + b) * (40 + 20 + 20)
    print(f"Online Appendix 10.B.4, alpha {a}, beta {b}: final stocks ({fmt(tgt, 1)}), funding ({fmt(tgt - [40, 20], 1)})")

Bs = np.linspace(0, 30, 301)
fig, ax = plt.subplots(figsize=(7.6, 3.6))
for (name, f), col, ls in zip(POLICIES.items(), ("#1F4E79", "#E65100", "#B71C1C"), ("-", "--", ":")):
    ax.plot(Bs, [f(B) for B in Bs], color=col, lw=2, ls=ls, label=f"{name}: {f(30.0):.1f} after 30")
ax.plot([tie], [ratios.max()], "o", color="#1F4E79", ms=5)
ax.annotate("all three tie at 90:\nthe bottleneck rotates", xy=(tie, ratios.max()), xytext=(19.5, 84.5), fontsize=8,
            color="#1F4E79", arrowprops=dict(arrowstyle="-", color="#1F4E79", lw=0.6))
ax.set_yticks([80, 85, 90, 95])
ax.set(xlabel="budget spent (index points)", ylabel="delivery capacity (index)", xlim=(0, 30), ylim=(78, 97.5),
       title="Funding the bottleneck buys the most capacity (seed 26110)")
ax.legend(frameon=False, fontsize=8, loc="upper left"); plt.tight_layout(); plt.show()

## Panel 8 — Audit the ledger (Theorems 10.1 and 10.5(iii); Online Appendix 10.B.5)

AXIOM Lab 1.10, step 5. Every stock obeys the balance of Equation 10.2: investment, use, decay, and conversion
in and out. Conversion leaks (Theorem 10.1(ii)): cash turned into a platform at 80 percent and the platform into
capability at 70 percent delivers **56 percent** (Online Appendix 10.B.1). The plug
$\pi_{i,k} = \Delta C_{i,k} - [\text{declared channels}]$ is zero exactly when the declared channels are complete and
correctly measured; one undeclared channel with flow $u$ shows up as $\pi = s\,u$, so its sign and timing locate it
(Theorem 10.5(iii)). Two of Meridian's records, built from the book's numbers and each missing one conversion:
the sale booked without the engineering team, and the proposed training cut booked as if the rent were paid.

In [ ]:
print(f"conversion chain: 0.80 x 0.70 = {0.80 * 0.70:.2f} of the budget arrives as capability; {1 - 0.80 * 0.70:.0%} leaks")

def audit(stocks, invest_declared, d=DELTA):
    # Theorem 10.5(iii): pi_k = C_{k+1} - C_k - [I_k - d C_k], the change the declared channels do not explain
    stocks = np.asarray(stocks, float)
    return np.diff(stocks) - (np.asarray(invest_declared, float) - d * stocks[:-1])

def report(name, ends, stocks, declared, cash_plug):
    # one row per quarter, labelled by the quarter it ends; capability in points, cash in $ million
    pi = audit(stocks, declared)
    print(name)
    print(f"  {'quarter':>7s} {'start':>8s} {'declared I':>11s} {'decay':>7s} {'end':>8s} {'plug':>7s} {'cash plug':>10s}")
    for k in range(len(pi)):
        print(f"  {ends[k]:7d} {stocks[k]:8.3f} {declared[k]:11.3f} {DELTA * stocks[k]:7.3f} {stocks[k + 1]:8.3f}"
              f" {round(pi[k], 9) + 0.0:7.2f} {cash_plug[k]:10.1f}")
    hit = np.nonzero(np.abs(pi) > 1e-9)[0]
    print(f"  nonzero plugs in quarters {[int(ends[k]) for k in hit]}: sign {'-' if pi[hit].sum() < 0 else '+'},"
          f" {abs(pi[hit].sum()):.2f} points in all; cash plug {sum(cash_plug):+.1f}")
    return pi

# Record A: the sale closes in quarter 0 and the team leaves with the unit; the ledger declares hiring,
# training and attrition, but not the team's transfer, and the treasury books the unit at the $92 million
# price it would fetch without the team
true_A = [RENT, RENT] + list(spend_full[:k58])
out_A = [0.0, POINTS_LOST] + [0.0] * k58
rec_A = [C_KEEP]
for i, u in zip(true_A, out_A):
    rec_A.append((1 - DELTA) * rec_A[-1] + i - u)
cash_A = [0.0, PROCEEDS_LEAVE - PROCEEDS_KEEP] + [0.0] * k58
pi_A = report("Record A: the sale, booked without the engineering team (quarter 0 ends with the closing)",
              np.arange(-1, k58 + 1), rec_A, true_A, cash_A)
print(f"  -> one outflow of {-pi_A.min():.0f} points on capability and ${max(cash_A):.0f} million into cash in the closing quarter:"
      f" a conversion at ${max(cash_A) / -pi_A.min():.0f} million a point, the team sold with the unit\n")

# Record B: training is cut for four quarters, but the ledger still books the rent as invested
true_B = [0.0] * CUT + [RENT] * CUT
declared_B = [RENT] * (2 * CUT)
rec_B, _ = path(C_KEEP, true_B, 2 * CUT)
cash_B = [PRICE * (dcl - tru) for dcl, tru in zip(declared_B, true_B)]
pi_B = report("Record B: the training cut, booked as if the rent were paid", np.arange(1, 2 * CUT + 1), rec_B, declared_B, cash_B)
print(f"  -> {-pi_B[0]:.2f} points a quarter out of capability and ${cash_B[0]:.1f} million a quarter into cash for exactly the four quarters"
      f" of the cut: the rent turned back into cash, ${sum(cash_B):.1f} million of reported earnings borrowed from the stock")

## Validation — the book's numbers, and agreement with `DCT_V1_Ch10_Lab.xlsx`

Each checkpoint is a number printed in Chapter 10 (Worked Example 10.1, Tables 10.1 and 10.5, Figures 10.1–10.3,
Section 10.7), in Online Appendix 10.B, or in Chapter 11's reading of the rent (Section 11.8). The workbook
recomputes the ones marked *workbook* with Excel formulas and shows PASS against the same engine values on its
`Checks` sheet.

In [ ]:
checks = [
    ("forms in the capital vector (workbook)", len(FORMS), 10, 0),
    ("price of a capability point (workbook)", PRICE, 4.0, 0.05),
    ("liquidity after the sale, team goes (workbook)", AFTER["let it go"][0], 732, 0.5),
    ("liquidity after the sale, team kept (workbook)", AFTER["keep the team"][0], 712, 0.5),
    ("technology after the sale (workbook)", TECH_AFTER, 36.9, 0.05),
    ("depreciation a year, percent (workbook)", 100 * (1 - (1 - DELTA) ** 4), 9.6, 0.05),
    ("rent on 58, points a quarter (workbook)", RENT, 1.45, 0.005),
    ("rent on 58, $ million a year (workbook)", 4 * PRICE * RENT, 23, 0.5),
    ("years to 90%, delta 5%, at the rent (workbook)", rental["organizational"][0], 44.9, 0.05),
    ("years to 90%, delta 10%, at the rent (workbook)", rental["human"][0], 21.9, 0.05),
    ("years to 90%, delta 25%, at the rent (workbook)", rental["technological"][0], 8.0, 0.05),
    ("years to 90%, delta 5%, twice the rent (workbook)", rental["organizational"][1], 11.7, 0.05),
    ("years to 90%, delta 10%, twice the rent (workbook)", rental["human"][1], 5.7, 0.05),
    ("years to 90%, delta 25%, twice the rent (workbook)", rental["technological"][1], 2.1, 0.05),
    ("ceiling under the cap (workbook)", CBAR, 100, 0),
    ("time to clear 55 at the cap, quarters (workbook)", t55, 1.72, 0.005),
    ("time to regain 58 at the cap, quarters (workbook)", t58, 4.4, 0.05),
    ("capability after one quarter at the cap (workbook)", full[1], 54.18, 0.005),
    ("capability after four quarters at the cap (workbook)", full[4], 57.53, 0.005),
    ("quarters below 55, rebuild at the cap (workbook)", base["q55"], 2, 0),
    ("quarters below 58, rebuild at the cap (workbook)", base["q58"], 5, 0),
    ("old budget: quarters below 55 (workbook)", k55m, 21, 0),
    ("old budget: capability after six years (workbook)", maint[24], 55.3, 0.05),
    ("old budget: quarters to reach 57 (workbook)", k57m, 64, 0),
    ("rebuild investment, points (workbook)", base["rebuild"], 11.91, 0.005),
    ("rebuild investment, $ million (workbook)", PRICE * base["rebuild"], 47.6, 0.05),
    ("keep investment, $ million (workbook)", PRICE * base["keep"], 29.0, 0.05),
    ("extra cost of rebuilding, $ million (workbook)", base["extra"], 18.6, 0.05),
    ("net proceeds, keep the team (workbook)", base["net_keep"], 63.0, 0.05),
    ("net proceeds, rebuild (workbook)", base["net_rebuild"], 64.4, 0.05),
    ("cap 2.0: quarter capability clears 55 (workbook)", sens["cap 2.0 points a quarter"]["q55"], 4, 0),
    ("cap 2.0: quarter capability regains 58 (workbook)", sens["cap 2.0 points a quarter"]["q58"], 9, 0),
    ("cap 2.0: extra cost, $ million (workbook)", sens["cap 2.0 points a quarter"]["extra"], 17.8, 0.05),
    ("attrition 6%: quarter capability clears 55 (workbook)", sens["attrition 6% a year"]["q55"], 2, 0),
    ("attrition 6%: quarter capability regains 58 (workbook)", sens["attrition 6% a year"]["q58"], 4, 0),
    ("attrition 6%: extra cost, $ million (workbook)", sens["attrition 6% a year"]["extra"], 19.4, 0.05),
    ("attrition 11%: quarter capability clears 55 (workbook)", sens["attrition 11% a year"]["q55"], 3, 0),
    ("attrition 11%: quarter capability regains 58 (workbook)", sens["attrition 11% a year"]["q58"], 6, 0),
    ("attrition 11%: extra cost, $ million (workbook)", sens["attrition 11% a year"]["extra"], 18.2, 0.05),
    ("$30m ask: keeping costs more, $ million (workbook)", ask30["net_rebuild"] - ask30["net_keep"], 11, 0.5),
    ("skipped rent: quarter below 55 (workbook)", below, 3, 0),
    ("repurchase of 2 points, exact (workbook)", repurchase(2.0, FLOOR)[0], 1.72, 0.005),
    ("repurchase of 2 points, approximation (workbook)", repurchase(2.0, FLOOR)[1], 1.78, 0.005),
    ("bottleneck: capacity after 30 (workbook)", POLICIES["fund the bottleneck"](30.0), 95.6, 0.05),
    ("equal split: capacity after 30 (workbook)", POLICIES["equal split"](30.0), 90.0, 0.05),
    ("all to human capital: capacity after 30 (workbook)", POLICIES["all to human capital"](30.0), 80.0, 0.05),
    ("shadow value before the tie (workbook)", lam[0], 0.57, 0.005),
    ("shadow value after the tie (workbook)", lam[1], 0.44, 0.005),
    ("budget at which all three tie (workbook)", tie, 17.5, 0),
    ("equal split: capacity a point (workbook)", eq_rate, 0.33, 0.005),
    ("OA 10.B.3: capacity with a budget of 20 (workbook)", l3, 20, 1e-9),
    ("OA 10.B.4: funding of form 1 (workbook)", (0.4 / 0.6) * 80 - 40, 13.3, 0.05),
    ("conversion chain, share delivered (workbook)", 0.80 * 0.70, 0.56, 1e-9),
]
ok, width = 0, max(len(name) for name, *_ in checks)
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol + 1e-9 else "FAIL"; ok += status == "PASS"
    print(f"{name:{width}s} {got:10.4f}   book {want:9.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 11 attaches performance to these stocks: the rate and efficiency at which a company converts
capital into declared outcomes, with the rent $\delta C^{\ast}$ as a required line in any honest performance
statement. AXIOM-10 runs the same model interactively (Lab 1.10); this notebook and the workbook are its
reproducible record.